# Execution Plan Analysis: Reading Physical Operators and Bottlenecks

**Module**: 10_performance_and_optimization  
**Notebook**: 02_explain_and_execution_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep reading of physical plans: Scan, Filter, Project, Exchange, Sort, HashAggregate, and join operators.
- **Why does this topic exist?**: Before touching cluster configurations, you must verify the query plan is optimal.
- **What problem does it solve?**: Missing that a query is triggering an unintended Cartesian product or full scan.
- **Where is it used in real-world Data Engineering?**: Authoring code reviews for high-throughput batch ETL pipelines.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Logical/physical plans, Spark UI |
| **Required depth** | Working knowledge: can read a physical plan and connect operators to bottlenecks |
| **Data Engineer** | Use `explain` output to identify scans, exchanges, joins, and aggregations. |
| **Senior Data Engineer** | Diagnose why an optimization did not apply and propose a rewrite. |
| **Spark specialist** | Reason about plan shape, statistics, and runtime AQE changes. |
| **Next topic** | `10_performance_and_optimization/03_query_optimization.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

`explain()` is the fastest way to see what Spark intends to do. It shows scans and pushed filters, join
strategies, exchanges, and aggregations. The plan is not a performance measurement, but it explains the shape of
the work; combining it with UI metrics tells you both what Spark does and what it costs.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Simple plan** | Unoptimized logical plan | Useful for seeing the original query structure |
| **Formatted plan** | Readable physical plan | Best for day-to-day inspection |
| **Cost plan** | Statistics-based plan | Shows estimated sizes and join costs when available |
| **Codegen plan** | Generated stages | Shows which operators are fused |
| **Scan** | Source read | Check format, filters, and partition pruning |
| **Exchange** | Shuffle | The main data-movement operator |

### How it works, step by step

1. Call `explain()` on the DataFrame or SQL query.
2. Read the plan from the bottom (sources) to the top (output).
3. Identify scans and pushed filters.
4. Identify joins and their strategies.
5. Count exchanges and note stage boundaries.
6. Compare with the UI to see actual row counts and timings.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Plan_Analysis_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
q = orders.filter("quantity > 3").groupBy("channel").agg(F.count("order_id").alias("cnt"))
q.explain("formatted")

---

## Experiment: Detecting Exchange Nodes in Complex Queries

- **1. What are we testing?**: Count shuffle exchange points in multi-step transformations.
- **2. What do we expect?**: Each groupBy, distinct, and join creates an Exchange operator.

In [ ]:
plan_text = q._jdf.queryExecution().executedPlan().toString()
print("Total Exchange (Shuffle) nodes:", plan_text.count("Exchange"))

**3. What happened?**: Counted physical Exchange nodes.  
**4. Why did it happen?**: Exchange operators represent network shuffle boundaries.  
**5. Practical Engineering Lesson**: Fewer Exchange nodes almost always means faster query execution.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Query review before production

- **Industry**: Financial services
- **Scenario**: A new risk query is about to be deployed to a shared cluster.
- **Problem**: A full scan or an unnecessary shuffle could affect other workloads.
- **Why this concept is relevant**: Reviewing the plan catches missing partition filters and a sort-merge join that could be broadcast.
- **Implementation idea**: Add plan checks to the review checklist and test with production-like statistics.
- **Expected result**: A query that scans only the required partitions and avoids a large shuffle.
- **Alternative**: Rely on production monitoring after deployment.
- **Trade-offs**: Monitoring catches problems after they affect users; plan review prevents them.

### Use Case 2: Explaining a regression

- **Industry**: E-commerce
- **Scenario**: A recommendation query slows down after a data growth spurt.
- **Problem**: The join strategy changed and an exchange now dominates.
- **Why this concept is relevant**: The plan shows the strategy change and the missing statistics that caused it.
- **Implementation idea**: Refresh statistics, compare plans, and decide whether to broadcast or repartition.
- **Expected result**: A plan that matches the new data size.
- **Alternative**: Add a hint to force the old strategy.
- **Trade-offs**: A hint may restore performance temporarily but can become wrong as data grows further.

---

## Business Value

- Fewer production surprises.
- Faster root-cause analysis.
- Better collaboration between SQL authors and platform engineers.
- Measured decisions instead of folklore.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Use `explain('formatted')` for readability.
- Check scans for `PushedFilters` and `PartitionFilters`.
- Count exchanges; each one is a potential bottleneck.
- Compare estimated and actual row counts.
- Re-check the plan after enabling AQE or changing statistics.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Read a plan bottom-up

**Steps**

1. Write a query with a filtered scan, a join, and an aggregation.
2. Print `explain('formatted')`.
3. Label each operator and identify the exchanges.

**What to observe**: The plan shows the order of operations and where data moves.

**Healthy result**: You can explain each operator's role.

**Problematic result**: A filter appears above the scan instead of inside it.

**Engineering takeaway**: Plan reading turns a query into a sequence of physical costs.

---

## Trade-offs and Alternatives

### Advantages

- No extra tooling
- Shows optimization decisions
- Works for SQL and DataFrames

### Limitations

- Estimates can be wrong
- Verbose
- Does not show runtime skew directly

### When to use it

- Query review
- Regression analysis
- Join debugging

### When not to use it

- Treating the plan as a benchmark
- Ignoring runtime metrics
- Reading only the final plan

### Alternatives inside Spark

- explain
- SQL tab
- AQE plan

### Alternatives outside Spark

- Query profilers
- Warehouse explain
- Plan visualizers

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify HashAggregate
**Problem Statement**: Find HashAggregate in explain output.  
**Input / Context**: q

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): BroadcastExchange Node
**Problem Statement**: Verify BroadcastExchange presence when broadcasting.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost Model Statistics
**Problem Statement**: Explain Statistics output in explain('cost').  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The missing partition filter

**Scenario**: A query filters by date but the plan shows a full scan.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the filter expression and the partition column type.
2. Look for a cast, function, or view that hides the partition column.
3. Rewrite the predicate to reference the partition column directly.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify HashAggregate):
```python
print('HashAggregate is visible in formatted plan.')
```

#### Solution 2 (BroadcastExchange Node):
```python
orders.join(F.broadcast(orders.limit(5)),'order_id').explain()
```

#### Solution 3 (Cost Model Statistics):
```python
print('Statistics show estimated row count and byte size at each node.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Plan Verification**: Verify filter pushdown, column pruning, and exchange count.
- **Formatted Plan**: Use explain('formatted') for clear node breakdowns.

### Essential Technical Terms
- `Exchange`: Shuffle operator.
- `HashAggregate`: In-memory hash aggregation.

### Common Engineering Mistakes
- **Running queries without checking plans**: Leads to surprise shuffles in production.

### Interview Questions on This Topic
- **Q: How do you know if an operator participates in WholeStageCodegen?**
  - *A*: It is prefixed with an asterisk (*) in the physical plan.


### More Interview Questions

- **Q: What should you look for first in a physical plan?**
  - *A*: Scans and exchanges: whether filters and partitions are pruned, and how many shuffles the query requires.
- **Q: Why can a plan look optimal but still be slow?**
  - *A*: The plan shows strategy, not runtime distribution; skew, slow sources, GC, or undersized tasks can dominate despite a good plan.

### What to Study Next
Open 03_query_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')